# DP and graph algorithms

Five DP problems and five graph algorithms, all implemented from scratch in pure Python.

## Dynamic programming

| Problem | State | Time | Space |
|---|---|---|---|
| LCS | table[i][j] = LCS(s1[:i], s2[:j]) | O(mn) | O(mn) |
| 0/1 Knapsack | dp[i][w] = max value, items 0..i-1, capacity w | O(nW) | O(nW) |
| Edit distance | dp[i][j] = Levenshtein(s1[:i], s2[:j]) | O(mn) | O(mn) |
| Matrix chain | cost[i][j] = min ops for subchain i..j | O(n^3) | O(n^2) |
| Coin change | dp[i] = min coins for sum i | O(nA) | O(A) |

## Graph algorithms

| Algorithm | Purpose | Time (adj list) | Handles negatives |
|---|---|---|---|
| BFS | SSSP (unit weights), reachability | O(V+E) | N/A |
| DFS | Reachability, finish times | O(V+E) | N/A |
| Dijkstra | SSSP, non-negative weights | O((V+E) log V) | No |
| Bellman-Ford | SSSP, detects negative cycles | O(VE) | Yes |
| Floyd-Warshall | All-pairs shortest paths | O(V^3) | Yes |

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np

from src.dp_algorithms import (
    longest_common_subsequence, lcs_backtrack,
    knapsack, edit_distance, matrix_chain_order, coin_change,
)
from src.graph_algorithms import (
    bfs, dfs, dijkstra, bellman_ford, floyd_warshall,
    shortest_path, fw_path,
)

## Longest common subsequence

Fill the DP table bottom-up: if s1[i-1] == s2[j-1], inherit the diagonal + 1; otherwise take the max of left/up. Backtrack to recover one LCS.

In [ ]:
s1, s2 = 'ABCBDAB', 'BDCABA'
length, table = longest_common_subsequence(s1, s2)
lcs = lcs_backtrack(s1, s2, table)

print(f's1 = {s1!r}')
print(f's2 = {s2!r}')
print(f'LCS length = {length}')
print(f'One LCS    = {lcs!r}')

# Visualize the DP table
arr = np.array(table)
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(arr, cmap='Blues')
ax.set_xticks(range(len(s2) + 1))
ax.set_yticks(range(len(s1) + 1))
ax.set_xticklabels([' '] + list(s2))
ax.set_yticklabels([' '] + list(s1))
for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        ax.text(j, i, arr[i, j], ha='center', va='center', fontsize=10)
ax.set_title('LCS DP table')
plt.tight_layout()
plt.show()

## 0/1 Knapsack

In [ ]:
weights = [2, 3, 4, 5, 9]
values  = [3, 4, 5, 8, 10]
capacity = 10

max_val, selected = knapsack(weights, values, capacity)
print(f'Max value: {max_val}')
print(f'Items taken: {selected} (weights {[weights[i] for i in selected]}, values {[values[i] for i in selected]})')
print(f'Total weight: {sum(weights[i] for i in selected)} / {capacity}')

## Edit distance (Levenshtein)

In [ ]:
pairs = [
    ('kitten', 'sitting'),
    ('saturday', 'sunday'),
    ('', 'abc'),
    ('abc', 'abc'),
]
for s1, s2 in pairs:
    dist, _ = edit_distance(s1, s2)
    print(f'{s1!r:>12} -> {s2!r:<12}: {dist} operations')

## Coin change

In [ ]:
for coins, amount in [([1, 5, 10, 25], 36), ([1, 5, 10, 25], 100), ([2], 3)]:
    min_coins, _ = coin_change(coins, amount)
    print(f'coins={coins}, amount={amount}: {min_coins} coins')

## Dijkstra vs Bellman-Ford

On positive-weight graphs, both return the same distances. Bellman-Ford is O(VE) vs Dijkstra's O((V+E) log V), but Bellman-Ford handles negative weights and detects negative cycles.

In [ ]:
graph = {
    'A': [('B', 1.0), ('C', 4.0)],
    'B': [('C', 2.0), ('D', 5.0)],
    'C': [('D', 1.0)],
    'D': [],
}

dj_dist, dj_pred = dijkstra(graph, 'A')
bf_dist, bf_pred, neg_cycle = bellman_ford(graph, 'A')

print('Distances from A:')
print(f'{'node':>6}  {'dijkstra':>10}  {'bellman-ford':>12}')
for node in sorted(dj_dist):
    print(f'{node:>6}  {dj_dist[node]:>10.1f}  {bf_dist[node]:>12.1f}')
print(f'\nNegative cycle detected: {neg_cycle}')

path = shortest_path(dj_pred, 'A', 'D')
print(f'Shortest path A->D: {" -> ".join(path)}')

## Floyd-Warshall all-pairs

Fills the distance matrix for all (i, j) pairs in O(V^3). After n^3 comparisons, dist[i][j] is the shortest path from i to j through any intermediate nodes.

In [ ]:
INF = float('inf')
matrix = [
    [0,   3,   INF, 7],
    [8,   0,   2,   INF],
    [5,   INF, 0,   1],
    [2,   INF, INF, 0],
]

dist, nxt = floyd_warshall(matrix)

print('All-pairs shortest distances:')
n = len(dist)
header = '    ' + ''.join(f'{j:>6}' for j in range(n))
print(header)
for i in range(n):
    row = ''.join(f'{dist[i][j]:>6.1f}' if dist[i][j] < INF else '   INF' for j in range(n))
    print(f'{i:>3} {row}')

path = fw_path(nxt, 0, 2)
print(f'\nShortest path 0->2: {path}')